In [5]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 2️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


import re
from collections import Counter
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool   # ⭐ 오늘의 주인공

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

✅ 로컬 Jupyter
✅ API 키 OK


In [6]:
def run_with_tools(question: str, tools: list, llm_obj=llm):
    """질문 + 도구 리스트를 받아 4단계 흐름을 실행하고 최종 답변 반환"""
    llm_with_tools = llm_obj.bind_tools(tools)
    messages = [HumanMessage(content=question)]

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        return response.content

    tool_map = {t.name: t for t in tools}
    for tc in response.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return llm_with_tools.invoke(messages).content

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


## 실습 A. 단위 변환기

In [7]:
class ConversionResult(BaseModel):
    original_value: float = Field(description="변환 전 값")
    original_unit: str = Field(description="변환 전 단위")
    converted_value: float = Field(description="변환 후 값")
    converted_unit: str = Field(description="변환 후 단위")

In [8]:
@tool
def convert_length(value: float, from_unit: str, to_unit: str) -> ConversionResult:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('m', 'cm', 'km', 'inch', 'ft')
        to_unit: 도착 단위 ('m', 'cm', 'km', 'inch', 'ft')
    """
    # 변환 로직
    value_to_cm = {"m" : 100, "cm" : 1, "km" : 100_000, "inch": 2.54, "feet":30.48}
    cm_to_value = {"m" : 0.01, "cm" : 1, "km" : 0.000001, "inch" : 0.3937, "feet" : 0.0328084}
    converted_value = value * value_to_cm[from_unit] * cm_to_value[to_unit]
    return ConversionResult(original_value=value, original_unit=from_unit, converted_value=converted_value, converted_unit=to_unit)



In [9]:
for q in ["100m는 몇 cm야?",
          "1cm는 몇 inch야?"]:
    print(f"❓ {q}")
    print(run_with_tools(q, [convert_length]))
    print()

❓ 100m는 몇 cm야?
100m는 10,000cm입니다.

❓ 1cm는 몇 inch야?
1cm는 약 0.3937 inch입니다.



In [14]:
from datetime import datetime

class DateDiff(BaseModel):
    date1: str = Field(description="년-월-일 형태의 문자열")
    date2: str = Field(description="년-월-일 형태의 문자열")
    days: int = Field(description="두 날짜 차이의 일수 값")
    weeks: int = Field(description="두 날짜 차이의 주수 값")

@tool
def days_between(date1: str, date2: str) -> DateDiff:
    """두 날짜(YYYY-MM-DD) 사이의 일수·주수를 반환합니다.s"""

    d1 = datetime.strptime(date1, '%Y-%m-%d')
    d2 = datetime.strptime(date2, '%Y-%m-%d')
    
    date_diff = abs((d2 - d1).days)
    weeks = date_diff // 7

    return DateDiff(date1=date1,date2=date2, days=date_diff, weeks=weeks)

In [15]:
for q in ["내 생일이 6월 30일인데, 현재 날짜인 10월 8일과 얼마나 지났는지 알려줘",
          "입과식이 8월 24일인데, 현재 날짜인 10월 8일과 얼마나 지났는지 알려줘"]:
    print("?? : ", q)
    print(run_with_tools(q, [days_between]))
    print()

?? :  내 생일이 6월 30일인데, 현재 날짜인 10월 8일과 얼마나 지났는지 알려줘
당신의 생일인 6월 30일에서 현재 날짜인 10월 8일까지 100일이 지났습니다. 이는 약 14주에 해당합니다.

?? :  입과식이 8월 24일인데, 현재 날짜인 10월 8일과 얼마나 지났는지 알려줘
2023년 8월 24일부터 2023년 10월 8일까지 총 45일이 지났습니다. 이는 약 6주에 해당합니다.

